# Chọn mô hình Fashion-MNIST cho demo

Chỉ train trên Colab GPU. Chạy `python -m scripts.dong_goi_colab` ở máy, upload `experiments/source.zip` qua Files. Các ô: nạp mã → train/so sánh → kiểm chứng/phân tích → xem kết quả → tải ZIP. Đây là bản dùng cấu trúc thư mục mới; kết quả cũ đã lưu vẫn giữ nguyên.


In [ ]:
import zipfile
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Chọn GPU T4 trước khi chạy."

PROJECT = Path("/content/fashion_selection")
PROJECT.mkdir(exist_ok=True)

with zipfile.ZipFile("/content/source.zip") as source:
    source.extractall(PROJECT)

RUNS = PROJECT / "experiments/runs"
print("GPU:", torch.cuda.get_device_name(0))

## Huấn luyện và lựa chọn

Ô này chạy lại toàn bộ thí nghiệm. Nếu đã có artifact, chuyển đến phần xem kết quả.

In [ ]:
import subprocess
import sys

with (
    subprocess.Popen(
        [sys.executable, "-u", "-m", "training.chon_mo_hinh"],
        cwd=PROJECT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process,
    open(PROJECT / "training.log", "w") as log,
):
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    assert process.wait() == 0, "Xem lỗi ở cuối log."

## Kiểm chứng và phân tích

Kiểm tra kỹ thuật trên GPU, sau đó phân tích checkpoint đã chốt. Không dùng kết quả test để điều chỉnh mô hình.

In [ ]:
for module in ["tests.kiem_tra_huan_luyen_gpu", "evaluation.phan_tich_ket_qua"]:
    with subprocess.Popen(
        [sys.executable, "-u", "-m", module],
        cwd=PROJECT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        assert process.wait() == 0, f"Kiểm tra lại {module}."

## Đọc kết quả

Validation dùng chọn cấu hình. Test dùng báo cáo cuối. Độ lệch chuẩn 3 seed mô tả mức dao động. Softmax cao vẫn có thể đoán sai.

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

RUN = RUNS / (RUNS / "latest_selection.txt").read_text().strip()
print("Run:", RUN.name)
display(pd.read_csv(RUN / "validation_comparison.csv"))
display(pd.read_csv(RUN / "test_comparison.csv"))
print(json.loads((RUN / "stability.json").read_text()))
display(Image(filename=str(RUN / "analysis/confident_errors.png")))
display(Image(filename=str(RUN / "analysis/train_class_means.png")))

## Tải kết quả

Lưu ZIP vào `experiments/runs/`. `demo_bundle/model.pt` là checkpoint đã chọn; giữ kèm cấu hình và preprocessing.

In [ ]:
from google.colab import files

RUN = RUNS / (RUNS / "latest_selection.txt").read_text().strip()
files.download(str(RUN) + ".zip")